# Training machine-learned potentials with autoplex_soap_turbo

A hands-on tutorial for someone who has never used atomate2, autoplex or this repository. By the end you will have:

1. understood what each tool does and how a training workflow is put together,
2. installed everything on the machines you use,
3. trained a Li–C–O potential **iteratively**, with your own random seed, in your own database,
4. pooled your DFT data with everyone else's in a shared database, **`lico_master`**, and trained a model from the pool,
5. changed how training configurations are generated (exercise: hybrid MC/MD),
6. used the potential in **LAMMPS with MADlib**, biasing a simulation towards an X-ray pattern,
7. run an **NMR-matching** simulation with ShiftML on a system ShiftML supports.

**Running every cell is safe.** Anything that touches a cluster, the queue or LAMMPS is off until you switch it on below. With the switches off, every cell still runs: the training loop runs here with its expensive stages replaced by stand-ins, and the shared database is simulated in memory. So you can work through the whole notebook first and then turn the switches on.

In [ ]:
RUN_SETUP = False   # Part 1: install on every machine, start the runner
SUBMIT = False      # Parts 2-4: put real flows in the queue, read real results
RUN_LAMMPS = False  # Parts 5-6: needs lmp built with MADlib

In [ ]:
import getpass
import logging
import os
import shlex
import subprocess
import textwrap
from pathlib import Path

import numpy as np

# This notebook lives in <repo>/tutorial/.
REPO = Path.cwd().resolve().parent
PY = REPO / "autoplex_venv" / "bin" / "python"
LICO = REPO / "workflows" / "lico"
RUNS = REPO / "tutorial" / "runs"
RUNS.mkdir(exist_ok=True)


def sh(cmd, act=True, cwd=REPO):
    """Run a bash command, stream its output, raise on a non-zero exit."""
    if not act:
        print("[not run]", cmd)
        return
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"exit {proc.returncode}: {cmd}")


print("repository:", REPO)

---
# Part 0: The ideas

## 0.1 Why train a potential, and why iteratively

Density-functional theory (DFT) gives accurate energies and forces, but a single calculation on a few dozen atoms takes minutes to hours. Molecular dynamics needs millions of force evaluations. A **machine-learned interatomic potential** is a fit to DFT that evaluates in microseconds. Here that fit is a **GAP** (Gaussian Approximation Potential): it compares each atom's environment, through *descriptors*, with environments it has seen in training, and interpolates.

A fit is only as good as its training set. It interpolates well and extrapolates badly. So the question is *which* configurations to spend DFT on. Guessing them in advance wastes DFT on what the model would have got right anyway, and misses what it gets wrong.

**Iterative training** lets the model choose:

![The iterative loop](figures/loop.svg)

1. Compute a small **seed** set with DFT and fit a first model.
2. Run **MD with that model**. It wanders into configurations the training set doesn't cover, which is exactly where it is unreliable.
3. **Select** the candidates least like anything in the training set.
4. Compute **only those** with DFT, add them, and refit.

Each round fixes the model where it was most wrong. That is the use case this repository is built for. The same loop applies whether the target is energies and forces (this tutorial) or dipoles (`workflows/water_dipole/`).

## 0.2 The tools, and who does what

![The software stack](figures/stack.svg)

| tool | what it is | what you touch |
|---|---|---|
| **MongoDB** | a database server. Each *project* gets a database holding its job queue, every job's output, and large files | a host, a user, a password |
| **jobflow** | a Python library for writing a computation as **Jobs** (functions decorated `@job`) whose inputs are other jobs' outputs, collected into a **Flow** | nothing directly. The workflows are written in it |
| **jobflow-remote** | the **runner**: a daemon that sends each Job to the **worker** (cluster) it names, over ssh, and moves results back | `jf flow list`, `jf job info` |
| **atomate2** | ready-made DFT jobs. "Run a VASP static on this structure" is one line, and the result comes back as a structured *task document* | nothing directly |
| **autoplex** | automated fitting of ML potentials from DFT data. The fork pinned here teaches it the `soap_turbo` descriptor turboGAP needs | the hyperparameter file |
| **autoplex_soap_turbo** | this repository: iterative loops built from the above, whose stages run on different clusters and whose potentials turboGAP can run | a settings file per workflow |
| **turboGAP** | a fast MD and Monte Carlo code for GAPs. It drives the sampling | `sampling.md` / `sampling.mc` keywords |
| **MADlib** | a LAMMPS package that runs turboGAP potentials and biases simulations towards experimental spectra | LAMMPS input decks |

## 0.3 A Flow in ten lines

Every workflow in this repository is a jobflow Flow. Here is the whole idea at toy size, run here. `add(1, 2)` doesn't compute anything when called: it returns a **Job**, and `.output` is a **reference** to a result that doesn't exist yet. Passing that reference to another job is what makes the second depend on the first.

In [ ]:
from jobflow import Flow, job, run_locally


@job
def add(a, b):
    return a + b


first = add(1, 2)
second = add(first.output, 10)          # depends on `first`, through its output
flow = Flow([first, second], output=second.output)
responses = run_locally(flow, log=False)
print("second job returned", responses[second.uuid][1].output)

Under jobflow-remote, `first` and `second` could run on different clusters. The runner moves the output between them through MongoDB.

## 0.4 One run, several machines

![Machines](figures/machines.svg)

The stages of one training run land on different clusters: VASP on one, `gap_fit` on another. **These machines share no filesystem.** That one fact explains most of the design:

* Nothing passes from one stage to the next as a file path, because the next stage's machine can't open it. Structures and potentials travel **through MongoDB** as payloads.
* Every machine has its **own copy** of this repository. After changing code you sync it everywhere (`setup_all_machines.sh --sync-only`), or the job that lands on another cluster runs old code.
* Settings name a **worker** per stage: `roihu_cpu_vasp`, `triton_gapfit`. A worker is `<machine>_<role>`, generated from `config/machines.conf`.

---
# Part 1: Installation

## 1.1 What you need first

* **Passwordless ssh** from the runner to every machine. Name a specific login node (`roihuc1`), not a round-robin alias, which can accept a connection and then stall.
* **CSC certificates last 24 hours.** Roihu needs a key plus a certificate signed in a browser. Renew it on your workstation and push it to the runner with `bash setup/refresh_csc_cert.sh`. When it lapses nothing reports an error: jobs just stop advancing.
* **A MongoDB database of your own**, and read/write on the shared `lico_master`. Whoever administers MongoDB creates them, for example in `mongosh`:

```js
use lico_alice
db.createUser({user: "alice", pwd: "...", roles: [
    {role: "dbOwner",   db: "lico_alice"},      // her own project: queue, outputs
    {role: "readWrite", db: "lico_master"}]})   // the shared pool (Part 3)
```

One database per person matters. `jf admin reset` (1.7) wipes the jobflow collections it finds, and the outputs go into collections called `outputs` and `outputs_blobs`. Two people sharing one database would mix their runs.

## 1.2 Clone, and build the local environment

```bash
git clone --recursive https://github.com/TiganyZ/autoplex_soap_turbo.git
cd autoplex_soap_turbo
bash setup/setup_env_uv.sh --work-dir "$PWD" --python 3.11 --roles runner
./autoplex_venv/bin/python -m ipykernel install --user --name autoplex_venv
jupyter lab tutorial/autoplex_soap_turbo_tutorial.ipynb
```

`--recursive` matters: `autoplex/` is a git submodule. **Then switch this notebook's kernel to `autoplex_venv`.** From here on the cells import this package.

Keep the checkout on local disk, not an Aalto home directory. A Python environment is about 28,000 files, and home volumes cap the *number* of files: the install dies with "Disk quota exceeded" while `df` shows space free.

In [ ]:
import autoplex
import autoplex_soap_turbo
import jobflow_remote

print("autoplex", autoplex.__version__, "| jobflow-remote", jobflow_remote.__version__)

## 1.3 You, your seed, your database, your machines

Your **seed** is what makes your run different from your colleagues'. It drives the random draws of the seed structures, the train/test split, where MD starts, and selection. **Pick a number nobody else in your group has.**

For testing, `MONGO["database"]` can be set to `atomate2`. The MongoDB user must be **authorised on that database**: being able to log in, or to read some other database, is not enough, and 1.4 checks this before anything is installed.

The machines are a real three-cluster layout: a workstation as runner, Roihu for VASP and turboGAP, Triton for `gap_fit`. Change hosts, accounts and directories to yours. Each `work_dir` is new, so the tutorial gets its own environments.

In [ ]:
USER = getpass.getuser()
SEED = 1                     # yours; different from everyone else's
PROJECT = f"lico_{USER}"     # jobflow-remote project: one project = one database

MONGO = {
    "host": "CHANGEME",
    "port": 27017,
    "database": PROJECT,     # for testing: "atomate2"
    "username": "CHANGEME",
    "auth_source": "",       # empty: authenticate against `database`
}
MASTER_DATABASE = "lico_master"

MACHINES = [
    {"name": "alto", "host": "alt", "roles": "runner",
     "work_dir": "/l/zarrout1/lico_tutorial", "python_version": "3.11"},
    {"name": "roihu-cpu", "host": "roihuc1", "roles": "vasp,turbogap",
     "work_dir": "/scratch/project_2003566/zarrout1/lico_tutorial",
     "scheduler": "slurm", "account": "project_2003566", "partition": "medium",
     "python_module": "python-data/3.12",
     "modules": "gcc/15.2.0,openmpi/5.0.10,openblas/0.3.30,netlib-scalapack/2.2.2",
     "max_jobs": 100, "vasp_cmd": "srun vasp_std"},
    {"name": "triton", "host": "tr", "roles": "gapfit",
     "work_dir": "/scratch/work/zarrout1/lico_tutorial",
     "scheduler": "slurm", "account": "sumo", "partition": "sumo",
     "python_module": "mamba", "modules": "triton/2025.1-gcc,gcc/13.3.0",
     "max_jobs": 10, "gap_fit_env": "/scratch/work/zarrout1/reece/dipole/test/env.sh"},
]

MONGO_READY = "CHANGEME" not in (MONGO["host"], MONGO["username"])
MONGO_PASSWORD = os.environ.get("TUTORIAL_MONGO_PASSWORD", "")
if MONGO_READY and not MONGO_PASSWORD:
    MONGO_PASSWORD = getpass.getpass("MongoDB password: ")


def machine(role):
    return next(m for m in MACHINES if role in m["roles"].split(","))


def worker(role):
    return f"{machine(role)['name'].replace('-', '_')}_{role}"


print(f"you: {USER}, seed {SEED}, project {PROJECT}")
print("workers:", [worker(r) for r in ("runner", "vasp", "turbogap", "gapfit")])

## 1.4 Can you reach both databases?

Check this first: every later failure caused by it looks like something else. A wrong password or a missing role shows here as `not authorized on <database>`, and not as a runner that never picks anything up.

In [ ]:
def check_database(mongo, password, database):
    import pymongo

    client = pymongo.MongoClient(
        host=mongo["host"], port=mongo["port"], username=mongo["username"],
        password=password, authSource=mongo["auth_source"] or mongo["database"],
        serverSelectionTimeoutMS=5000)
    probe = client[database]["tutorial_write_probe"]
    probe.insert_one({"probe": True})
    probe.drop()
    print(f"read/write OK on '{database}'")


if MONGO_READY:
    check_database(MONGO, MONGO_PASSWORD, MONGO["database"])
    check_database(MONGO, MONGO_PASSWORD, MASTER_DATABASE)
else:
    print("[not run] fill in MONGO above first")

## 1.5 Write the machine configuration

You edit one file, `config/machines.conf`. Everything else is generated from it: the jobflow-remote project, the per-machine environment, and the atomate2 settings. The tutorial writes its own copy, `config/machines.tutorial.conf`, so an existing setup is untouched. It holds the password, so it is owner-readable and gitignored. Every key is listed in `docs/training-guide.md` §2.

In [ ]:
CONF = REPO / "config" / "machines.tutorial.conf"


def machines_conf(project, mongo, password, machines):
    lines = [f"project_name={project}",
             f"mongodb_host={mongo['host']}",
             f"mongodb_port={mongo['port']}",
             f"mongodb_database={mongo['database']}",
             f"mongodb_username={shlex.quote(mongo['username'])}",
             f"mongodb_password={shlex.quote(password)}",
             f"mongodb_auth_source={mongo['auth_source']}", ""]
    for m in machines:
        keys = [f"{k}={shlex.quote(str(v))}" for k, v in m.items() if k != "name"]
        lines.append(f"machine {m['name']} \\\n    " + " \\\n    ".join(keys))
    return "\n".join(lines) + "\n"


if MONGO_READY:
    CONF.touch(mode=0o600, exist_ok=True)
    CONF.chmod(0o600)
    CONF.write_text(machines_conf(PROJECT, MONGO, MONGO_PASSWORD, MACHINES))
    print("wrote", CONF)
else:
    print(machines_conf(PROJECT, MONGO, "<password>", MACHINES))
sh(f"git check-ignore -q {CONF} && echo 'gitignored: yes'")

## 1.6 Install on every machine

On each machine, `setup_all_machines.sh` does the following:

1. rsyncs this repository to `<work_dir>/autoplex_soap_turbo`,
2. builds a Python environment, with autoplex only where a role needs it,
3. builds turboGAP on `turbogap` machines, and QUIP on `gapfit` machines that have no `gap_fit_env`,
4. writes `<work_dir>/env.sh`, which every job sources.

Then it installs the jobflow-remote project on the runner as `~/.jfremote/<project>.yaml`. It is safe to re-run, `--only <machine>` limits it, and `DRY_RUN=1` only prints. The first run takes tens of minutes, mostly building turboGAP.

Rendering writes `config/generated/`, which your main configuration shares. Afterwards, `bash setup/setup_all_machines.sh --config-only` restores your main project's files.

In [ ]:
sh(f"DRY_RUN=1 bash setup/setup_all_machines.sh --config {CONF}", act=MONGO_READY)
sh(f"bash setup/setup_all_machines.sh --config {CONF}", act=RUN_SETUP)
sh(f"bash setup/check_setup.sh --config {CONF}", act=RUN_SETUP)

## 1.7 Start the runner

`jf admin reset -f` initialises your new database, and **deletes every flow already in it without asking**. That is fine on a fresh database and never right on a shared one. `jf runner start` starts the daemon. Installing the project file locally lets this notebook submit and read results; both only need MongoDB.

In [ ]:
RUNNER = machine("runner")
on_runner = f"ssh {RUNNER['host']} 'source {RUNNER['work_dir']}/env.sh && "
sh(on_runner + f"jf -p {PROJECT} admin reset -f && jf -p {PROJECT} runner start'", act=RUN_SETUP)
sh(on_runner + f"jf -p {PROJECT} project check --errors'", act=RUN_SETUP)
sh(f"mkdir -p ~/.jfremote && install -m 600 config/generated/jobflow_remote.yaml "
   f"~/.jfremote/{PROJECT}.yaml", act=RUN_SETUP)

---
# Part 2: Iterative training with your own seed

## 2.1 The workflow

`workflows/lico/` trains an energy/force GAP for Li₂CO₃-stoichiometry Li–C–O against VASP:

| file | what it holds |
|---|---|
| `training.yaml` | the run: iterations, workers, the INCAR, the sampler, selection |
| `gap_hypers.yaml` | the descriptors: only `soap_turbo`, `distance_2b` and `angle_3b`, the ones turboGAP evaluates natively |
| `data/make_seed.py` | the seed cells: crystalline Li₂CO₃ (C2/c), strained, sheared and rattled |
| `run.py` | three lines: hand the settings to the shared front end |

The seed carries no DFT labels, so the flow runs VASP on it first. Each iteration then does fit → turboGAP MD with that fit → select → VASP → merge, and the last iteration only fits. The code is `src/autoplex_soap_turbo/flows/iterative_energy.py`.

In [ ]:
print((LICO / "training.yaml").read_text())

## 2.2 Your seed, your settings

Your copy of the settings changes two things: the seed file, and `dataset.seed`. Both are files of your own (`training_user_<you>.yaml`, `data/seed_<you>.xyz`), and both are gitignored.

In [ ]:
def user_settings(user, seed):
    """training.yaml with this user's seed file and seed."""
    text = (LICO / "training.yaml").read_text()
    text = text.replace("initial: data/seed.xyz", f"initial: data/seed_{user}.xyz")
    text = text.replace("  seed: 0\n", f"  seed: {seed}\n")
    path = LICO / f"training_user_{user}.yaml"
    path.write_text(text)
    return path


sh(f"{PY} workflows/lico/data/make_seed.py --seed {SEED} --out workflows/lico/data/seed_{USER}.xyz")
MY_SETTINGS = user_settings(USER, SEED)
RUN = f"{PY} workflows/lico/run.py --config {MY_SETTINGS.relative_to(REPO)}"
print("settings:", MY_SETTINGS.relative_to(REPO))

## 2.3 Check it before anything is submitted

Three levels, cheapest first:

* **`--dry-run`** builds the flow and prints every job with its worker. Read the worker column: a stage on the wrong machine is what this catches.
* **`--stub`** *runs* the whole flow here, in seconds. VASP, `gap_fit` and turboGAP are replaced by stand-ins returning outputs of the right shape (`src/autoplex_soap_turbo/stubs.py`). Everything between them is real: the dataset split, selection, merging, the payloads crossing every job boundary, the summary.
* A real run.

In [ ]:
sh(RUN + " --dry-run")

The same stub run, done in Python rather than through `run.py`, so we keep its job store and can read results out of it exactly as you will from MongoDB. `memory_job_store()` stands in for your database.

In [ ]:
from autoplex_soap_turbo.flows.iterative_energy import (
    EnergyTrainingConfig,
    iterative_energy_training,
)
from autoplex_soap_turbo.stubs import memory_job_store, stubbed_energy_stages

for name in ("jobflow", "autoplex_soap_turbo"):
    logging.getLogger(name).setLevel(logging.WARNING)

def stub_run(settings_file, root):
    """Run a flow here with VASP, gap_fit and turboGAP faked; return (flow, store)."""
    settings = EnergyTrainingConfig.from_file(settings_file)
    store = memory_job_store()
    Path(root).mkdir(parents=True, exist_ok=True)
    with stubbed_energy_stages():
        flow = iterative_energy_training(settings)
        run_locally(flow, store=store, log=False, create_folders=True, root_dir=root,
                    ensure_success=True)
    return flow, store


flow, my_store = stub_run(MY_SETTINGS, RUNS / "stub_mine")
summary = flow.output.resolve(my_store)
for row in summary["iterations"]:
    print(row)

The training set grows every iteration while the test set stays fixed. New frames go to training only, so every iteration is scored on the same held-out frames and the errors can be compared. (With stubs, the errors are invented.)

## 2.4 Submit, and watch

Sync first: every machine imports this package from its **own** copy.

In [ ]:
sh(f"bash setup/setup_all_machines.sh --config {CONF} --sync-only", act=SUBMIT)
sh(RUN + f" --project {PROJECT}", act=SUBMIT)

```bash
jf -p lico_<you> flow list                 # your flows and their ids
jf -p lico_<you> job list -fid <flow-id>   # every job and its state
jf -p lico_<you> job info <db-id>          # including the traceback, if it failed
jf -p lico_<you> job rerun -did <db-id>    # after fixing and --sync-only
```

A run with this setup is a handful of VASP batches (16 seed cells, then 10 per iteration, each 15–60 minutes on 64–128 cores) and four short fits. Expect most of a day of queueing. `docs/running.md` covers triage. On CSC machines the classic silent stall is an expired certificate: jobs sit in `CHECKED_OUT` and nothing says why.

## 2.5 Reading your results

The summary is the flow's output, and each fit's potential is in its job's output. `get_jobstore` connects to your project's database. With `SUBMIT` off, the stub's store stands in.

In [ ]:
import matplotlib.pyplot as plt


def run_summary(store, run_name):
    doc = store.query_one({"name": f"{run_name}: summary"}, properties=["output"])
    return doc["output"]


if SUBMIT:
    from jobflow_remote import get_jobstore

    store = get_jobstore(project_name=PROJECT)
    store.connect()
else:
    store = my_store

rows = run_summary(store, "lico")["iterations"]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot([r["n_train"] for r in rows], [r["test_rmse_energy_meV_per_atom"] for r in rows], "o-")
ax.set_xlabel("training frames")
ax.set_ylabel("test RMSE (meV/atom)")
ax.set_title("one point per iteration")
plt.show()

---
# Part 3: Pooling everyone's DFT into `lico_master`

![Pooling](figures/master.svg)

Everyone ran the same loop with a different seed, so everyone's DFT covers somewhat different configurations. All of it was paid for. Pooled, it trains a better model than any single run.

`autoplex_soap_turbo.data.master` does the pooling:

| call | what it does |
|---|---|
| `harvest_run(project, run_name)` | every labelled frame of a finished run: the newest merged dataset, seed included, train and test |
| `MasterDatabase.from_project(project, database="lico_master")` | the shared database, on your project's MongoDB server with your credentials |
| `master.push(frames, provenance)` | one document per configuration, tagged with user, project, run and seed. A configuration already present is not stored twice |
| `master.pull(criteria)` | the pooled frames. It **refuses to mix DFT settings**: each frame carries a hash of its INCAR's energy-relevant keys and POTCARs, and a fit across two settings learns the difference between them, not the chemistry |
| `master.summary()` | who contributed how much |

## 3.1 Push your data

With `SUBMIT` off, this simulates the classroom. Three people, one of them you, each run the stubbed loop with their own seed, and push into an in-memory `lico_master`.

In [ ]:
from maggma.stores import MemoryStore

from autoplex_soap_turbo.data.master import MasterDatabase, frames_from_store, harvest_run

if SUBMIT:
    master = MasterDatabase.from_project(PROJECT, database=MASTER_DATABASE)
    mine = harvest_run(PROJECT, "lico")
    print(master.push(mine, {"user": USER, "project": PROJECT, "run_name": "lico",
                             "seed": SEED}))
else:
    master = MasterDatabase(MemoryStore(collection_name="configurations"))
    classmates = {USER: (SEED, my_store)}
    for name, seed in (("alice", 7), ("bob", 8)):
        sh(f"{PY} workflows/lico/data/make_seed.py --seed {seed} "
           f"--out workflows/lico/data/seed_{name}.xyz")
        _, their_store = stub_run(user_settings(name, seed), RUNS / f"stub_{name}")
        classmates[name] = (seed, their_store)
    for name, (seed, their_store) in classmates.items():
        counts = master.push(frames_from_store(their_store, "lico"),
                             {"user": name, "project": f"lico_{name}", "run_name": "lico",
                              "seed": seed})
        print(f"{name:>8}: {counts}")

The first person adds every frame. Later people add fewer: the unrattled, strained crystals are identical for every seed, so they are already there. Duplicates are recognised by a hash of the species, positions and cell, together with the DFT settings.

In [ ]:
for contributor, n in master.summary().items():
    print(f"{contributor:>20}: {n} configurations")

## 3.2 Train a model from the pool

The pooled frames carry DFT labels, so the same flow fits **straight away**, with no seed VASP pass. `iterations: 1` is fit-only. Raise it and the pooled model carries on iterating from there.

One limit to know about: the seed frames travel inside the job document, which MongoDB caps at 16 MB. That is a few thousand cells of this size. Past that, read the data inside a job instead.

In [ ]:
from autoplex_soap_turbo.data.dataset import write_dataset

pooled = master.pull()
master_file = write_dataset(LICO / "data" / "master.xyz", pooled)
print(f"{len(pooled)} pooled frames -> {master_file.relative_to(REPO)}")

text = (LICO / "training.yaml").read_text()
text = text.replace("name: lico\n", "name: lico_master_fit\n")
text = text.replace("initial: data/seed.xyz", "initial: data/master.xyz")
text = text.replace("iterations: 4", "iterations: 1")
MASTER_SETTINGS = LICO / "training_user_master.yaml"
MASTER_SETTINGS.write_text(text)

master_flow, master_store = stub_run(MASTER_SETTINGS, RUNS / "stub_master")
print(master_flow.output.resolve(master_store)["iterations"])
sh(f"{PY} workflows/lico/run.py --config {MASTER_SETTINGS.relative_to(REPO)} --project {PROJECT}",
   act=SUBMIT)

---
# Part 4: Exercise — your own configuration-generation protocol

MD at 800 K explores near the crystal. Suppose you want the cell to **change volume** too, and to sample with **hybrid MC/MD**: short MD bursts that are accepted or rejected as a whole, mixed with Monte-Carlo volume moves at a set pressure. What do you change?

**Start by asking whether it is a setting or code.** turboGAP already implements Monte-Carlo walks with these move types (`docs/keywords.md` in turboGAP):

| move | does |
|---|---|
| `md` | redraws velocities, runs `md_nsteps` of MD, then accepts or rejects the result. `mc_hamiltonian` puts the kinetic energy in the test, and that only balances with `velocity_distribution: maxwell` |
| `volume` | rescales the cell by up to `mc_lnvol_max` in ln V, at pressure `p_beg`, **in bar** |
| `move`, `swap`, `insertion`, `removal`, `relax` | single-atom moves, species swaps, grand-canonical exchange, relaxation |

In this repository a turboGAP Monte-Carlo walk is `sampling.method: gcmc` (the name dates from its first use). `mc_types` and `mc_acceptance` choose the moves, and `sampling.mc` is copied verbatim into turboGAP's input. **So this is a settings change.**

### Your turn

Write the `sampling:` section below. Hints:
* keep `worker` and `resources` as they were,
* no `insertion` or `removal`: nothing is exchanged,
* turboGAP warns if `p_beg` is left at its default of 1 bar, and if `md` moves have no `thermostat`.

In [ ]:
MY_SAMPLING = """
sampling:
  worker: roihu_cpu_turbogap
  resources: {account: project_2003566, partition: small, nodes: 1,
              ntasks_per_node: 8, time: "01:00:00"}
  method: turbogap_md     # TODO: change to a Monte-Carlo walk with md and volume moves
  n_candidates: 60
  md:
    md_nsteps: 4000
    md_step: 0.5
    thermostat: bussi
    t_beg: 800.0
    t_end: 800.0
    write_xyz: 50
"""


def with_sampling(sampling_block, out):
    """training.yaml with its sampling section replaced."""
    text = (LICO / "training.yaml").read_text()
    start, end = text.index("sampling:\n"), text.index("selection:\n")
    text = text[:start] + textwrap.dedent(sampling_block).strip() + "\n\n" + text[end:]
    text = text.replace("initial: data/seed.xyz", f"initial: data/seed_{USER}.xyz")
    out.write_text(text)
    return out


exercise = with_sampling(MY_SAMPLING, LICO / "training_user_exercise.yaml")
sh(f"{PY} workflows/lico/run.py --config {exercise.relative_to(REPO)} --dry-run | head -8")

### Solution

`workflows/lico/training_hybrid.yaml` is the answer. Here is how it differs from `training.yaml`:

In [ ]:
import difflib

diff = difflib.unified_diff((LICO / "training.yaml").read_text().splitlines(),
                            (LICO / "training_hybrid.yaml").read_text().splitlines(),
                            "training.yaml", "training_hybrid.yaml", lineterm="", n=0)
print("\n".join(diff))

In [ ]:
sh(f"{PY} workflows/lico/run.py --config workflows/lico/training_hybrid.yaml --stub | tail -n 12")

`tests/test_iterative_energy.py::test_the_hybrid_walk_hands_turbogap_md_and_volume_moves` intercepts the walk and checks what turboGAP would be given: `n_mc_types` before `mc_types`, because turboGAP allocates the list from it, `p_beg`, and no exchange lists.

### When it is not a setting

If turboGAP had no such move, you would write a sampler. In this flow the seam is small:

1. **The simulation**, as a plain function in `src/autoplex_soap_turbo/turbogap/`, with no jobflow inside, so it can be tested and reused outside a flow.
2. **The dispatch** in `sample_periodic` (`flows/iterative_energy.py`): a branch on `sampling.method`, the new name added to `SAMPLERS`, and to the `allowed` set in `SamplingSettings` (`config.py`).
3. **Strip what the model wrote.** A sampler's frames carry the model's own energies and forces, under names the next fit would read as DFT data. `strip_model_outputs` and `strip_reference_labels` remove them. Nothing reports it if you forget: the model agrees with itself and the errors improve.
4. **Four tests**, the same ones `docs/custom-workflows.md` §7 asks for: what it does (call `sample_periodic.original(...)`), that the flow's arguments serialise, that labels are stripped, and that the stage lands on its worker.
5. **Sync to every machine** before submitting.

`docs/custom-workflows.md` is the full guide, including how to write a flow of a different shape.

---
# Part 5: Your potential in LAMMPS, matched to an X-ray pattern

![MAD](figures/mad.svg)

**MADlib** is a LAMMPS package. Two of its pieces matter here:

* `pair_style turbogap` evaluates a turboGAP `.gap` potential, all four terms (soap_turbo, 2b, 3b, core repulsion), as turboGAP itself would.
* A `fix mad/*` adds **(γ/2)·D²** to the energy, where D measures the distance between a spectrum predicted from the current atoms and a target. The bias has forces, so dynamics are pushed towards structures that reproduce the measurement. `mad/xrd` and `mad/pdf` do diffraction, `mad/nmr` does NMR, `mad/xps` does photoelectron spectra.

The test here: take crystalline Li₂CO₃, compute its X-ray pattern as the "measurement", **scramble the atoms**, and anneal with and without the bias. If the bias works, the biased run's relative error `f_xrd[2]` falls well below the control's.

## 5.1 Build LAMMPS with MADlib

MADlib's installer builds LAMMPS, QUIP, symmetrix (MACE in LAMMPS) and MADlib under `PREFIX`, and a Python environment there. Each stage is skipped once its output exists, so re-running resumes a failed build. Put `PREFIX` on local or scratch disk.

In [ ]:
MADLIB_SRC = Path.home() / "src" / "madlib"        # a checkout of github.com/TiganyZ/madlib
PREFIX = Path.home() / "work" / "turbogap-lammps"
LMP = PREFIX / "lammps" / "build" / "lmp"
LMP_PYTHON = PREFIX / "venv" / "bin" / "python"   # the Python lmp embeds
MAD_DECKS = REPO / "tutorial" / "madlib"

sh(f"git clone https://github.com/TiganyZ/madlib {MADLIB_SRC}",
   act=RUN_LAMMPS and not MADLIB_SRC.exists())
sh(f"MADLIB_SOURCE={MADLIB_SRC} PREFIX={PREFIX} JOBS=8 "
   f"bash {MADLIB_SRC}/tools/install_turbogap_lammps.sh", act=RUN_LAMMPS and not LMP.exists())
sh(f"{LMP} -h | grep -oE 'mad/(xrd|pdf|nmr|property/python)|turbogap|symmetrix/mace' | sort -u",
   act=RUN_LAMMPS)

## 5.2 Your potential, converted for turboGAP

The final fit's potential is a payload in your database. `payload_to_files` writes the GAP XML and its sparse-point files. turboGAP and MADlib read a converted `.gap` file, produced by turboGAP's `make_gap_files.py`, so convert **once**, on a machine that has turboGAP (`TURBOGAP_ROOT`). The `.gap` names its data files by relative path: keep the whole `gap_files/` directory together.

`pair_style turbogap` takes `e0` per species as an input parameter, because the `.gap` file does not carry it. Energies are offset by a constant without it, and forces are unaffected.

In [ ]:
from autoplex_soap_turbo.payload import main_file, payload_to_files

TURBOGAP_ROOT = Path.home() / "turbogap"          # a turboGAP checkout, built
XRD_DIR = RUNS / "xrd"
XRD_DIR.mkdir(parents=True, exist_ok=True)
LICO_GAP = XRD_DIR / "gap_files" / "LiCO" / "LiCO.gap"


def fetch_last_fit(store, run_name, out_dir):
    """Write the newest fit's GAP XML (and sparse files) to out_dir."""
    docs = list(store.query({"name": {"$regex": f"^{run_name}: fit \\d+$"}},
                            properties=["uuid", "output.iteration"]))
    latest = max(docs, key=lambda d: d["output"]["iteration"])
    potential = store.query_one({"uuid": latest["uuid"]}, load=True)["output"]["potential"]
    payload_to_files(potential, out_dir)
    return Path(out_dir) / main_file(potential, ".xml")


if SUBMIT and RUN_LAMMPS:
    from autoplex_soap_turbo.turbogap.potential import convert_potential

    xml = fetch_last_fit(store, "lico", XRD_DIR / "xml")
    os.environ["TURBOGAP_ROOT"] = str(TURBOGAP_ROOT)
    LICO_GAP, dropped = convert_potential(xml, ["Li", "C", "O"], XRD_DIR, "LiCO")
    print(LICO_GAP, "dropped:", dropped)
else:
    print("[not run] needs a finished real run; the potential goes to", LICO_GAP)

## 5.3 The target: crystalline Li₂CO₃'s X-ray pattern

A `gamma 0`, `run 0` pass computes the pattern and writes it. The `mad/*` target reader takes the first two columns, so that output is itself a target. `fix mad/xrd` requires a target even at γ = 0, so a flat placeholder on the same q range stands in.

In [ ]:
import importlib.util

from ase.io import write as ase_write


def lico_seed_module():
    spec = importlib.util.spec_from_file_location("lico_make_seed", LICO / "data" / "make_seed.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def write_lammps(path, atoms):
    ase_write(path, atoms, format="lammps-data", specorder=["Li", "C", "O"], masses=True)


crystal = lico_seed_module().li2co3().repeat((1, 2, 2))
scrambled = crystal.copy()
scrambled.set_chemical_symbols(np.random.default_rng(SEED).permutation(crystal.get_chemical_symbols()))
write_lammps(XRD_DIR / "crystal.data", crystal)
write_lammps(XRD_DIR / "scrambled.data", scrambled)
q = np.linspace(0.5, 9.0, 200)
np.savetxt(XRD_DIR / "flat_target.dat", np.c_[q, np.ones_like(q)])
for deck in ("in.xrd_target", "in.xrd_anneal"):
    (XRD_DIR / deck).write_text((MAD_DECKS / deck).read_text())
print(f"{len(crystal)}-atom cell; species shuffled in the scrambled copy")
print((MAD_DECKS / "in.xrd_target").read_text())

sh(f"{LMP} -in in.xrd_target -var data crystal.data -var out li2co3_xrd.dat -log target.log",
   act=RUN_LAMMPS, cwd=XRD_DIR)

## 5.4 Anneal, with and without the bias

**Choosing γ:** run once with γ = 0 and read `f_xrd[3]`, which is D. Pick γ so that γD/2 is a small fraction of the cohesive energy. Here it ramps from 0 to γ over the run, so the bias switches on gently.

In [ ]:
print((MAD_DECKS / "in.xrd_anneal").read_text())


def xrd_anneal(tag, gamma, nsteps=4000, seed=4928):
    sh(f"{LMP} -in in.xrd_anneal -var data scrambled.data -var pot {LICO_GAP} "
       f"-var target li2co3_xrd.dat -var gam {gamma} -var seed {seed} -var nsteps {nsteps} "
       f"-var tag {tag} -log {tag}.log", act=RUN_LAMMPS, cwd=XRD_DIR)


xrd_anneal("xrd_control", 0.0)
xrd_anneal("xrd_biased", 20.0)

In [ ]:
def last_block(path):
    """The final block a fix mad/* wrote: (rel_error, x, y_pred, y_target)."""
    block = Path(path).read_text().split("# step")[-1].splitlines()
    words = block[0].split()
    header = dict(zip(words[1::2], words[2::2]))
    data = np.array([[float(v) for v in line.split()] for line in block[1:] if line.strip()])
    return float(header["rel_error"]), data[:, 0], data[:, 1], data[:, 2]


if RUN_LAMMPS:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for tag in ("xrd_control", "xrd_biased"):
        error, x, pred, target = last_block(XRD_DIR / f"{tag}_xrd.dat")
        ax.plot(x, pred, label=f"{tag} (rel. error {error:.2f})")
    ax.plot(x, target, "k--", label="crystal")
    ax.set_xlabel("q (1/Å)")
    ax.set_ylabel("q F(q)")
    ax.legend()
    plt.show()

To run this on a cluster instead, put the same `lmp -in in.xrd_anneal -var ...` line in a Slurm script, with `srun` in front for MPI, on a machine where `PREFIX` is built. MADlib's fixes run in parallel, and `comm_modify cutoff` in the deck covers the reach of `compute mad/pdf`.

---
# Part 6: NMR matching with ShiftML

ShiftML predicts NMR chemical shieldings from structure, and `compute mad/property/python mad_shiftml` makes those predictions available to MADlib. `fix mad/nmr` then biases towards a target spectrum, one fix per nucleus.

**ShiftML has no lithium.** ShiftML 4.1 is fitted for H, C, N, O, F, Na, Mg, P, S, Cl, K and Ca, and refuses any structure containing Li. Li₂CO₃ is therefore not a ShiftML system. Matching it would need a shielding model fitted for Li–C–O, through `compute mad/property/turbogap`. The system that works is **succinic acid** (C₄H₆O₄, 28 atoms per cell), whose data ships with MADlib.

The MD needs a potential for H, C and O. **MACE-MP-0b2 small** is a public (MIT) foundation model that `pair_style symmetrix/mace` runs. The original MACE-MP-0 small can't be used, because symmetrix only converts models whose first interaction block is non-residual.

## 6.1 The potential: download and convert

symmetrix reads its own `.json` format. `tutorial/madlib/extract_mace.py` converts a MACE model with symmetrix's extractor, for the elements you name. It needs torch, e3nn and mace-torch, which the installer's Python environment has.

In [ ]:
NMR_DIR = RUNS / "nmr"
NMR_DIR.mkdir(parents=True, exist_ok=True)
MACE_URL = ("https://github.com/ACEsuit/mace-mp/releases/download/mace_mp_0b2/"
            "mace-small-density-agnesi-stress.model")
MACE_MODEL = NMR_DIR / "mace-mp-0b2-small.model"
MACE_JSON = NMR_DIR / "mace_mp0b2_small_HCO.json"
EXTRACTOR = PREFIX / "symmetrix" / "symmetrix" / "source" / "symmetrix" / "extract_mace_data.py"

sh(f"curl -sSL -o {MACE_MODEL} {MACE_URL}", act=RUN_LAMMPS and not MACE_MODEL.exists())
sh(f"{LMP_PYTHON} {MAD_DECKS / 'extract_mace.py'} {EXTRACTOR} {MACE_MODEL} {MACE_JSON} 1 6 8",
   act=RUN_LAMMPS and not MACE_JSON.exists())
sh(f"{LMP_PYTHON} -m pip install shiftml", act=RUN_LAMMPS)

## 6.2 The environment ShiftML needs inside LAMMPS

A LAMMPS compute passes no arguments to its Python module, so ShiftML is configured by environment variables. `LD_PRELOAD` works around a real crash: lmp links the system OpenBLAS, `libtorch_cpu` defines some of the same symbols, and if OpenBLAS is loaded first, torch's first batched matrix multiply segfaults.

In [ ]:
def nmr_env():
    if not RUN_LAMMPS:
        return "<ShiftML environment> "
    torch_lib = subprocess.run(
        [str(LMP_PYTHON), "-c", "import os, torch; print(os.path.dirname(torch.__file__))"],
        capture_output=True, text=True, check=True).stdout.strip()
    return (f"export LD_PRELOAD={torch_lib}/lib/libtorch_cpu.so "
            "MADLIB_SHIFTML_TYPES='H C O' MADLIB_SHIFTML_MODEL=ShiftML41 "
            "MADLIB_SHIFTML_DEVICE=cpu && ")


if RUN_LAMMPS:
    for name in ("mad_shiftml.py",):
        (NMR_DIR / name).write_text((MADLIB_SRC / "tests" / name).read_text())
    for name in ("succinic.data", "succinic_shuffled.data"):
        (NMR_DIR / name).write_text((MADLIB_SRC / "tests" / "data" / name).read_text())
for deck in ("in.nmr_predict", "in.nmr_anneal"):
    (NMR_DIR / deck).write_text((MAD_DECKS / deck).read_text())

## 6.3 Target spectra from the crystal

Predict the pristine crystal's shieldings, then broaden them into ¹H and ¹³C targets on the same grids and widths the fixes use (`tutorial/madlib/nmr_target.py`). Both sides are ShiftML shieldings, so the fixes use `convention shift`. Against a *measured* spectrum, you would instead give `reference`/`slope` (δ = a − bσ) to turn shieldings into shifts: `fix mad/nmr` refuses to run until one or the other is set.

In [ ]:
TARGET = MAD_DECKS / "nmr_target.py"
sh(nmr_env() + f"{LMP} -in in.nmr_predict -var data succinic.data -var out pristine_cs.dump -log none",
   act=RUN_LAMMPS, cwd=NMR_DIR)
sh(f"{PY} {TARGET} pristine_cs.dump 1 5.0 40.0 141 1.5 nmr_H.dat", act=RUN_LAMMPS, cwd=NMR_DIR)
sh(f"{PY} {TARGET} pristine_cs.dump 2 -40.0 200.0 241 4.0 nmr_C.dat", act=RUN_LAMMPS, cwd=NMR_DIR)

## 6.4 Rebuild the crystal from its spectrum

`succinic_shuffled.data` has its atoms shuffled onto one another's sites. Anneal it from 1000 K to 300 K, biased (γ = 10) and as a control (γ = 0).

* **σ sets the stiffness.** The bias force scales as 1/σ². 1.5 ppm (¹H) and 4 ppm (¹³C) are workable; at 0.5 ppm a scrambled cell saw 131 eV/Å.
* **NVT at the experimental cell.** In earlier runs a MACE foundation model expanded this molecular crystal by over 50% at 1 bar, so a barostat would undo the comparison.
* **Time:** roughly an hour per run on one CPU at `nsteps 6000`; almost all of it is ShiftML.

In [ ]:
print((MAD_DECKS / "in.nmr_anneal").read_text())


def nmr_anneal(tag, gamma, nsteps=6000, seed=4928):
    sh(nmr_env() + f"{LMP} -in in.nmr_anneal -var data succinic_shuffled.data "
       f"-var mace {MACE_JSON} -var gam {gamma} -var nsteps {nsteps} -var seed {seed} "
       f"-var tag {tag} -log {tag}.log", act=RUN_LAMMPS, cwd=NMR_DIR)


nmr_anneal("nmr_control", 0.0)
nmr_anneal("nmr_biased", 10.0)

In [ ]:
if RUN_LAMMPS:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, nucleus, label in zip(axes, ("H", "C"), ("$^1$H", "$^{13}$C"), strict=True):
        for tag in ("nmr_control", "nmr_biased"):
            error, x, pred, target = last_block(NMR_DIR / f"{tag}_{nucleus}.spec")
            ax.plot(x, pred, label=f"{tag} ({error:.2f})")
        ax.plot(x, target, "k--", label="crystal")
        ax.set_xlabel("shielding (ppm)")
        ax.set_title(label)
    axes[0].legend()
    plt.show()

---
# Where next

* **Every settings key:** `docs/settings-reference.md`. `sampling.md`, `sampling.mc` and `vasp.user_incar_settings` pass straight through to turboGAP and VASP.
* **Dipoles and IR spectra:** `workflows/water_dipole/` and `workflows/lif/` run the same loop for dipole models.
* **A flow of a different shape:** `docs/custom-workflows.md`.
* **DFT you already have on disk:** `docs/building-a-training-set.md`.
* **A submitted run that misbehaves:** `docs/running.md`.